**Part 2 · LangChain** · step 10 of 25  
[← 09 Tools & tool calling](09_tools_and_tool_calling.ipynb) · [🏠 Roadmap](../README.md) · [11 Agent memory →](11_agent_memory.ipynb)

# 10 · Your First Agent — `create_agent`

In notebook 09 you wrote the tool loop yourself: call model → run tools → call model again.
An **agent** automates that loop. `create_agent` builds one in a single call
(it is a ready-made LangGraph graph under the hood — you'll build the same thing by hand in notebook 18).

**What you'll learn**
- What an *agent* is: **LLM + tools + a loop** that keeps calling tools until it can answer.
- How to build one in a single call with `create_agent`.
- How to read the agent's output (a `dict` holding the full message history).

**Mental model**
```
user message ──► LLM ──(wants a tool?)──yes──► run tool ──► result back to LLM ──┐
                  ▲                                                              │
                  └──────────────────────────────────────────────────────────────┘
                  └──no──► final answer
```

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## Define a tool and create the agent

Any plain Python function can be a tool. The **function name, type hints and docstring**
are turned into a schema the LLM reads to decide *when* and *how* to call it —
so a clear docstring is not optional, it is the tool's instructions.

In [ ]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city, e.g. "Delhi".

    Returns:
        A short weather description (mocked here, a real tool would call an API).
    """
    print("I am being called")  # proves the agent really executed the tool
    return f"Great weather for {city}"


agent = create_agent(
    model="gpt-5",                 # "provider:model" or just a model name (OpenAI is inferred)
    tools=[get_weather],           # tools the LLM is allowed to call
    system_prompt="You are a helpful assistant",
)
agent  # in Jupyter this renders the underlying LangGraph graph

## Run the agent

Input and output are both **state dicts** with a `"messages"` key.
Messages can be given as dicts (`{"role": ..., "content": ...}`) or as message objects.

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "whats the weather for Delhi"}]}
)

I am being called


In [ ]:
# The last message is the agent's final answer
response["messages"][-1].content

### See every step the agent took
`pretty_print()` shows the full loop: Human → AI (tool call) → Tool result → AI (final answer).

In [ ]:
for message in response["messages"]:
    message.pretty_print()

## Revision notes
| Concept | Remember |
|---|---|
| `create_agent(model, tools, system_prompt)` | One-line ReAct-style agent (built on LangGraph under the hood) |
| Tool = function + docstring | The docstring/type hints become the schema the LLM sees |
| `agent.invoke({"messages": [...]})` | Returns a **dict**, not a message — the whole conversation |
| `response["messages"][-1].content` | The final answer |

---
✅ **Done with 10 First agent.** Next: [11 Agent memory →](11_agent_memory.ipynb)  
[🏠 Roadmap](../README.md)